In [1]:
%pip install -U langgraph


Note: you may need to restart the kernel to use updated packages.


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
import re
from pydantic import BaseModel

In [3]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

api_key = getpass("Cole sua OpenAI API Key: ").strip()

os.environ["OPENAI_API_KEY"] = api_key

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=api_key
)

In [4]:
class State(TypedDict, total=False):
    proposal: str
    topic: str
    relevance_score: float
    methodology_score: float
    feasibility_score: float
    impact_score: float
    final_score: float | None
    status: str


class Avaliacao(BaseModel):
    nota: float
    justificativa: str


avaliador = llm.with_structured_output(Avaliacao)


In [5]:
#Funções de Avaliação
#Esta célula define as funções utilizadas no processo de avaliação, incluindo a extração de pontuações e os componentes individuais de avaliação.

In [33]:
def extract_score(content: str) -> float:
    """Extrai a pontuação numérica da resposta do LLM."""
    match = re.search(r'Pontuação:\s*(\d+(?:[.,]\d+)?)', content)

    if match:
        score = float(match.group(1).replace(",", "."))

        if 0 <= score <= 1:
            return score

    raise ValueError(f"Não foi possível extrair uma pontuação válida de: {content}")

def check_relevance(state: State) -> State:
    """Verifica a relevância da proposta científica."""

    prompt = ChatPromptTemplate.from_template(
        "Analise a relevância da seguinte proposta de pesquisa "
        "em relação ao tema: {topic}. "
        "Considere a importância do problema e a clareza dos objetivos. "
        "Forneça uma pontuação de relevância entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )

    result = llm.invoke(
        prompt.format(
            topic=state["topic"],
            proposal=state["proposal"]
        )
    )

    try:
        state["relevance_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_relevance: {e}")
        state["relevance_score"] = 0.0

    return state

    try:
        state["relevance_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_relevance: {e}")
        state["relevance_score"] = 0.0

    return state

def check_methodology(state: State) -> State:
    """Avalia o rigor metodológico da proposta científica."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a metodologia da seguinte proposta de pesquisa. "
        "Considere o rigor científico, a fundamentação teórica, "
        "a adequação dos métodos e a coerência com os objetivos. "
        "Forneça uma pontuação de metodologia entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )

    result = llm.invoke(
        prompt.format(proposal=state["proposal"])
    )

    try:
        state["methodology_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_methodology: {e}")
        state["methodology_score"] = 0.0

    return state
    
def check_feasibility(state: State) -> State:
    """Avalia a viabilidade de execução do projeto científico."""
    prompt = ChatPromptTemplate.from_template(
        "Analise a viabilidade da seguinte proposta de pesquisa. "
        "Considere os recursos necessários, as limitações técnicas, "
        "a complexidade de implementação e a possibilidade de execução. "
        "Forneça uma pontuação de viabilidade entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )
    result = llm.invoke(prompt.format(
        proposal=state["proposal"]
    ))
    try:
        state["feasibility_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em check_feasibility: {e}")
        state["feasibility_score"] = 0.0
    return state

def evaluate_impact(state: State) -> State:
    """Avalia o impacto científico esperado da proposta."""
    prompt = ChatPromptTemplate.from_template(
        "Avalie o impacto científico da seguinte proposta de pesquisa. "
        "Considere a originalidade, a contribuição para o conhecimento, "
        "o potencial de inovação e a relevância dos resultados esperados. "
        "Forneça uma pontuação de impacto entre 0 e 1. "
        "Sua resposta deve começar com 'Pontuação: ' seguida da pontuação numérica, "
        "depois forneça sua explicação.\n\nProposta: {proposal}"
    )
    result = llm.invoke(prompt.format(
        proposal=state["proposal"]
    ))
    try:
        state["impact_score"] = extract_score(result.content)
    except ValueError as e:
        print(f"Erro em evaluate_impact: {e}")
        state["impact_score"] = 0.0
    return state

def calculate_final_score(state: State) -> State:
    """Calcula a pontuação final da proposta científica."""
    state["final_score"] = (
        state["relevance_score"] * 0.20 +
        state["methodology_score"] * 0.35 +
        state["feasibility_score"] * 0.25 +
        state["impact_score"] * 0.20
    )
    return state

    if any(state.get(c) is None for c in criterios):
        return {
            "final_score": None,
            "status": "Avaliação interrompida"
        }

   
    return {
        "final_score": final_score,
        "status": "Avaliação completa"
    }

In [34]:
print(check_methodology)

<function check_methodology at 0x000001D5FDCFA5C0>


In [35]:
#Definição do fluxo de trabalho
#Essa célula define o fluxo de trabalho de avaliação usando StateGraph.

In [36]:
workflow = StateGraph(State)

# Adiciona nós
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("check_methodology", check_methodology)
workflow.add_node("check_feasibility", check_feasibility)
workflow.add_node("evaluate_impact", evaluate_impact)
workflow.add_node("calculate_final_score", calculate_final_score)

# Define arestas condicionais
workflow.add_conditional_edges(
    "check_relevance",
    lambda x: (
        "check_methodology"
        if x["relevance_score"] > 0.5
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "check_methodology",
    lambda x: (
        "check_feasibility"
        if x["methodology_score"] > 0.6
        else "calculate_final_score"
    )
)

workflow.add_conditional_edges(
    "check_feasibility",
    lambda x: (
        "evaluate_impact"
        if x["feasibility_score"] > 0.7
        else "calculate_final_score"
    )
)
workflow.add_conditional_edges(
    "evaluate_impact",
    lambda x: (
    "calculate_final_score"
    )
)


# Entrada e saída
workflow.set_entry_point("check_relevance")
workflow.add_edge("calculate_final_score", END)

# Compila
app = workflow.compile()

In [37]:
def grade_proposal(proposal: str, topic: str) -> dict:
    """Avalia a proposta científica usando o fluxo de trabalho definido."""
    initial_state = State(
        proposal=proposal,
        topic=topic,
        relevance_score=0.0,
        methodology_score=0.0,
        feasibility_score=0.0,
        impact_score=0.0,
        final_score=None,
        status="Pendente"
    )
    result = app.invoke(initial_state)
    return result

    

In [ ]:
sample_proposal = """
O avanço da Inteligência Artificial (IA) tem aumentado a necessidade de recursos computacionais para o processamento de dados e a execução de algoritmos complexos. Nesse contexto, a computação quântica surge como uma alternativa promissora, utilizando princípios da mecânica quântica, como superposição e emaranhamento, para desenvolver novas estratégias de processamento. A integração dessas tecnologias poderá contribuir para determinadas aplicações de aprendizado de máquina e otimização computacional.

O objetivo desta pesquisa é investigar o potencial da computação quântica em aplicações de Inteligência Artificial, analisando o desempenho de algoritmos quânticos e híbridos quântico-clássicos em comparação com métodos tradicionais de aprendizado de máquina.

A metodologia será desenvolvida utilizando Python e ferramentas de computação quântica, como Qiskit ou PennyLane. Inicialmente, serão estudados os fundamentos da computação quântica e sua aplicação em algoritmos de aprendizado de máquina. Posteriormente, serão implementados algoritmos de classificação quântica, como Variational Quantum Classifier (VQC) e métodos baseados em kernels quânticos, utilizando conjuntos de dados públicos de pequena dimensão. Os resultados serão comparados com algoritmos clássicos, como Support Vector Machines (SVM) e redes neurais artificiais, considerando métricas de acurácia, precisão, F1-score e tempo de processamento. Também serão investigados os efeitos da quantidade de qubits, da profundidade dos circuitos e do ruído quântico no desempenho dos modelos.

Como resultados esperados, pretende-se identificar as vantagens e limitações dos algoritmos quânticos em tarefas específicas de Inteligência Artificial, avaliando sua capacidade de classificação e seus requisitos computacionais. A pesquisa também buscará compreender em quais condições abordagens híbridas quântico-clássicas podem apresentar resultados competitivos em relação aos métodos tradicionais.

Por fim, este projeto pretende contribuir para a compreensão das possibilidades de integração entre computação quântica e Inteligência Artificial. Por meio de experimentos computacionais, será possível investigar o desempenho de algoritmos quânticos, suas limitações e sua viabilidade para aplicações práticas. A principal contribuição científica será uma análise comparativa entre métodos clássicos e quânticos, considerando desempenho, complexidade e eficiência computacional, sem pressupor a existência de vantagem quântica generalizada.

"""



result = grade_proposal(
    sample_proposal,
    "Computadores Quânticos em Aplicações com Inteligência Artificial"
)

final_score = result["final_score"] * 10
relevance_score = result["relevance_score"] * 10
methodology_score = result["methodology_score"] * 10
feasibility_score = result["feasibility_score"] * 10
impact_score = result["impact_score"] * 10

print("Relevância:", result["relevance_score"])
print("Metodologia:", result["methodology_score"])
print("Viabilidade:", result["feasibility_score"])
print("Impacto:", result["impact_score"])
print("Nota final:", result["final_score"])

Pontuação Final: 7.88/10

Relevância: 9.40/10
Metodologia: 7.80/10
Viabilidade: 8.60/10
Impacto: 5.60/10
